### Importation des bibliothèques

### Installation des d?pendances (Kaggle-safe)


In [ ]:
import sys
import subprocess
import importlib.util


def ensure_package(import_name: str, pip_spec: str):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_spec}")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            pip_spec,
        ])


# Kaggle-safe: do not force-upgrade torch/cuda.
ensure_package("numpy", "numpy")
ensure_package("scipy", "scipy")
ensure_package("pandas", "pandas")
ensure_package("sklearn", "scikit-learn")
ensure_package("yaml", "pyyaml")
ensure_package("gensim", "gensim")
ensure_package("tqdm", "tqdm")
ensure_package("torch", "torch")

print("Dependencies check done")


In [1]:
import os
import scipy.io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import normalized_mutual_info_score
import time
import random
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

### Configuration de l'environnement

In [ ]:
from pathlib import Path


def detect_project_root():
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path.cwd().resolve() / "Projet_PPD",
        Path.cwd().resolve().parent / "Projet_PPD",
    ]
    for candidate in candidates:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Impossible de trouver le dossier projet contenant data/ et output/.")


PROJECT_ROOT = detect_project_root()
DATA_ROOT = str(PROJECT_ROOT / 'data')
SAVE_ROOT = str(PROJECT_ROOT / 'output' / 'HyperMiner')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASETS = ["20NG", "IMDB", "YahooAnswer", "AGNews"]
K_VALUES = [50, 100]
SEED = 42

if not os.path.exists(SAVE_ROOT):
    os.makedirs(SAVE_ROOT)


def set_seed(seed=2022):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    print(f"Graine fix?e ? : {seed}")


set_seed(SEED)


### Chargement des données et fonctions métriques d'évaluation

In [3]:
def load_dataset_(name):
    path = os.path.join(DATA_ROOT, name)
    train = sp.load_npz(os.path.join(path, "train_bow.npz")).toarray()
    test = sp.load_npz(os.path.join(path, "test_bow.npz")).toarray()
    vocab = [w.strip() for w in open(os.path.join(path, "vocab.txt"), encoding="utf-8")]
    test_labels = np.loadtxt(os.path.join(path, "test_labels.txt"), dtype=int)
    return train, test, vocab, test_labels

def get_metrics(beta, theta, y_test, vocab):
    # Topic Diversity
    num_topics = beta.shape[0]
    list_w = []
    for k in range(num_topics):
        idx = beta[k].argsort()[-10:][::-1]
        list_w.extend([vocab[i] for i in idx])
    td = len(set(list_w)) / len(list_w)
    
    # Clustering (NMI)
    preds = theta.argmax(1)
    nmi = normalized_mutual_info_score(y_test, preds)
    
    # Purity
    y_voted = np.zeros(y_test.shape)
    labels = np.unique(y_test)
    for cluster in np.unique(preds):
        mask = (preds == cluster)
        if mask.any():
            hist, _ = np.histogram(y_test[mask], bins=np.append(labels, labels.max() + 1))
            y_voted[mask] = labels[np.argmax(hist)]
    purity = np.mean(y_voted == y_test)
    
    return td, nmi, purity

### Implémentation du modèle HyperMiner

In [4]:
class HyperMiner_Model(nn.Module):
    def __init__(self, vocab_size, num_topics, embed_size=50, c=-0.01):
        super().__init__()
        self.num_topics = num_topics
        self.curvature = torch.tensor([c]).to(DEVICE)
        
        # Initialisation standard
        self.rho = nn.Parameter(torch.empty(vocab_size, embed_size).normal_(std=0.02))
        self.alpha = nn.Parameter(torch.empty(num_topics, embed_size).normal_(std=0.02))
        
        self.encoder = nn.Sequential(
            nn.Linear(vocab_size, 300),
            nn.ReLU(),
            nn.Linear(300, 300),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.fc_mu = nn.Linear(300, num_topics)
        self.fc_logvar = nn.Linear(300, num_topics)

    def poincare_dist(self, x, y, c):
        dot_xy = torch.mm(x, y.t())
        norm_x_sq = torch.sum(x**2, dim=-1, keepdim=True)
        norm_y_sq = torch.sum(y**2, dim=-1, keepdim=True).t()
        num = norm_x_sq - 2*dot_xy + norm_y_sq
        den = (1 + c * norm_x_sq) * (1 + c * norm_y_sq)
        return num / (den + 1e-7)

    def get_beta(self, temp=0.05):
        dist = self.poincare_dist(self.alpha, self.rho, self.curvature)
        return F.softmax(-dist / temp, dim=-1)

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + torch.randn_like(std) * std
        return mu

    def forward(self, x, kl_w, temp=0.05):
        denorm = torch.where(x.sum(1, keepdim=True) > 0, x.sum(1, keepdim=True), torch.tensor([1.0]).to(DEVICE))
        x_norm = x / denorm
        h = self.encoder(x_norm)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        theta = F.softmax(z, dim=-1)
        beta = self.get_beta(temp=temp)
        recon = torch.mm(theta, beta)
        
        recon_loss = -(x * (recon + 1e-10).log()).sum(1).mean()
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1).mean()
        return recon_loss + (kl_w * kl_loss), recon_loss, kl_loss

### Boucle d'entraînement du modèle HyperMiner

In [5]:
def train_hyperminer(dataset_name, K):
    train_bow, _, vocab, _ = load_dataset_(dataset_name)
    model = HyperMiner_Model(len(vocab), K).to(DEVICE)
    
    # Weight decay intermédiaire pour stabiliser le TD
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-5)
    
    loader = DataLoader(TensorDataset(torch.from_numpy(train_bow).float()), batch_size=200, shuffle=True)
    
    for epoch in tqdm(range(150), desc=f"HyperMiner Balanced {dataset_name} K={K}", leave=False):
        model.train()
        kl_w = min(1.0, epoch / 50.0)
        for batch in loader:
            x = batch[0].to(DEVICE)
            optimizer.zero_grad()
            # On entraîne avec la même température cible
            loss, _, _ = model(x, kl_w, temp=0.05)
            loss.backward()
            optimizer.step()
            
    return model

### Évaluation des performances et calcul des métriques

In [ ]:
CHECKPOINT_PATH = os.path.join(SAVE_ROOT, "hyperminer_fast_results.csv")
TIMING_PATH = os.path.join(SAVE_ROOT, "HyperMiner_training_times.csv")

all_hyper_results = pd.read_csv(CHECKPOINT_PATH).to_dict('records') if os.path.exists(CHECKPOINT_PATH) else []
all_timing_rows = pd.read_csv(TIMING_PATH).to_dict('records') if os.path.exists(TIMING_PATH) else []

for dataset_name in DATASETS:
    print(f"\n DATASET : {dataset_name}")
    train_bow, test_bow, vocab, y_test = load_dataset_(dataset_name)

    for K in K_VALUES:
        if any(r['Dataset'] == dataset_name and r['K'] == K for r in all_hyper_results):
            print(f" K={K} d?j? calcul?, skip.")
            continue

        print(f" Entraînement K={K}...")
        total_start = time.perf_counter()

        train_start = time.perf_counter()
        model = train_hyperminer(dataset_name, K)
        train_seconds = time.perf_counter() - train_start
        model.eval()

        with torch.no_grad():
            beta = model.get_beta().cpu().numpy()
            test_tensor = torch.from_numpy(test_bow).float().to(DEVICE)
            h = model.encoder(test_tensor / (test_tensor.sum(1, keepdim=True) + 1e-10))
            theta = F.softmax(model.fc_mu(h), dim=-1).cpu().numpy()

        td, nmi, pur = get_metrics(beta, theta, y_test, vocab)
        total_seconds = time.perf_counter() - total_start

        res = {"Dataset": dataset_name, "K": K, "TD": round(td, 3), "Purity": round(pur, 3), "NMI": round(nmi, 3), "Time": round(total_seconds, 2)}
        all_hyper_results.append(res)
        pd.DataFrame(all_hyper_results).to_csv(CHECKPOINT_PATH, index=False)

        trow = {
            'model': 'HyperMiner', 'dataset': dataset_name, 'K': int(K), 'seed': int(SEED), 'device': str(DEVICE),
            'phase': 'train', 'train_seconds': float(train_seconds), 'total_seconds': float(total_seconds),
            'train_minutes': float(train_seconds / 60.0), 'total_minutes': float(total_seconds / 60.0),
        }
        all_timing_rows.append(trow)
        pd.DataFrame(all_timing_rows).to_csv(TIMING_PATH, index=False)

        print(f" K={K} | TD: {td:.3f} | Pur: {pur:.3f} | NMI: {nmi:.3f} | Time={total_seconds:.2f}s")

print("\n--- R?SULTATS HYPERMINER ---")
print(pd.DataFrame(all_hyper_results).to_string(index=False))

if all_timing_rows:
    df_h_t = pd.DataFrame(all_timing_rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    print("\n--- TIMING HYPERMINER ---")
    print(df_h_t[['dataset', 'K', 'device', 'train_seconds', 'total_seconds']].to_string(index=False))
    print(f"Saved training times: {TIMING_PATH}")


### Calcul de la Cohérence Sémantique

In [7]:
# --- CONFIGURATION ---
CV_HYPER_SAVE_PATH = os.path.join(SAVE_ROOT, "hyperminer_cv_results.csv")
TOPN = 15

# 1. Chargement des scores existants
if os.path.exists(CV_HYPER_SAVE_PATH):
    hyper_cv_results = pd.read_csv(CV_HYPER_SAVE_PATH).to_dict('records')
    print(f"{len(hyper_cv_results)} scores C_V chargés depuis le checkpoint.")
else:
    hyper_cv_results = []

print("CALCUL C_V GENSIM")

for dataset_name in DATASETS:
    needed_ks = [k for k in K_VALUES if not any(r['Dataset'] == dataset_name and r['K'] == k for r in hyper_cv_results)]
    
    if not needed_ks:
        print(f"{dataset_name} | Déjà calculé.")
        continue

    print(f"\n" + "="*50 + f"\n DATASET : {dataset_name}\n" + "="*50)
    
    # Reconstruction du corpus complet
    train_bow, _, vocab_list, _ = load_dataset_(dataset_name)
    print(f" Reconstruction des textes ({train_bow.shape[0]} documents)...")
    
    texts = []
    for i in range(train_bow.shape[0]):
        idx = train_bow[i].nonzero()[0]
        if len(idx) > 1:
            texts.append([vocab_list[w] for w in idx])
    
    dictionary = Dictionary(texts)
    
    for K in needed_ks:
        print(f"\n HyperMiner | K={K} | Entraînement et calcul C_V...")
        
        # 2. Entraînement avec les paramètres validés (Balanced)
        model = train_hyperminer(dataset_name, K)
        model.eval()
        
        with torch.no_grad():
            # Beta extrait via la distance hyperbolique
            beta = model.get_beta(temp=0.05).cpu().numpy()
        
        # 3. Extraction des thèmes pour Gensim
        topics_ids = []
        for k in range(K):
            idx = beta[k].argsort()[-TOPN:][::-1]
            words = [vocab_list[i] for i in idx]
            ids = [dictionary.token2id[w] for w in words if w in dictionary.token2id]
            if len(ids) >= 2:
                topics_ids.append(ids)
        
        # 4. Calcul de la cohérence
        cv_score = 0.0
        if topics_ids:
            try:
                cm = CoherenceModel(
                    topics=topics_ids, 
                    texts=texts, 
                    dictionary=dictionary, 
                    coherence='c_v'
                )
                cv_score = cm.get_coherence()
            except Exception as e:
                print(f"Erreur Gensim : {e}")
        
        # 5. Sauvegarde 
        res_cv = {"Dataset": dataset_name, "K": K, "C_V": round(cv_score, 4)}
        hyper_cv_results.append(res_cv)
        pd.DataFrame(hyper_cv_results).to_csv(CV_HYPER_SAVE_PATH, index=False)
        
        print(f"{dataset_name} K={K} | C_V = {cv_score:.4f}")

# --- SYNTHÈSE FINALE ---
df_cv_hyper = pd.DataFrame(hyper_cv_results)
print("\n" + "#"*50 + "\n RÉCAPITULATIF COHÉRENCE HYPERMINER\n" + "#"*50)
if not df_cv_hyper.empty:
    print(df_cv_hyper.pivot(index="Dataset", columns="K", values="C_V"))

8 scores C_V chargés depuis le checkpoint.
CALCUL C_V PALMETTO
20NG | Déjà calculé.
IMDB | Déjà calculé.
YahooAnswer | Déjà calculé.
AGNews | Déjà calculé.

##################################################
 RÉCAPITULATIF COHÉRENCE HYPERMINER
##################################################
K               50      100
Dataset                    
20NG         0.4228  0.4169
AGNews       0.4055  0.4575
IMDB         0.3825  0.3257
YahooAnswer  0.5922  0.5189


### Analyse comparative : Résultats originaux vs Reproduction

In [8]:
import pandas as pd

# 1. Données du Papier Original 
data_paper = {
    'Dataset': ['20NG', '20NG', 'IMDB', 'IMDB', 'YahooAnswer', 'YahooAnswer', 'AGNews', 'AGNews'],
    'K': [50, 100, 50, 100, 50, 100, 50, 100],
    'CV_pap': [0.371, 0.368, 0.347, 0.343, 0.344, 0.346, 0.359, 0.360],
    'TD_pap': [0.613, 0.446, 0.485, 0.258, 0.507, 0.444, 0.521, 0.343],
    'Purity_pap': [0.433, 0.454, 0.655, 0.641, 0.456, 0.448, 0.730, 0.679],
    'NMI_pap': [0.405, 0.386, 0.046, 0.032, 0.237, 0.222, 0.276, 0.200]
}

# 2. Données de votre Reproduction
data_repro = {
    'Dataset': ['20NG', '20NG', 'IMDB', 'IMDB', 'YahooAnswer', 'YahooAnswer', 'AGNews', 'AGNews'],
    'K': [50, 100, 50, 100, 50, 100, 50, 100],
    'CV_rep': [0.423, 0.417, 0.406, 0.458, 0.383, 0.326, 0.592, 0.519],
    'TD_rep': [0.604, 0.486, 0.606, 0.510, 0.624, 0.530, 0.662, 0.595],         
    'Purity_rep': [0.544, 0.614, 0.678, 0.628, 0.570, 0.553, 0.808, 0.766],   
    'NMI_rep': [0.546, 0.552, 0.061, 0.038, 0.334, 0.307, 0.380, 0.316]        
}

df_pap = pd.DataFrame(data_paper)
df_rep = pd.DataFrame(data_repro)

# Fusion des deux tableaux
df_compare = pd.merge(df_pap, df_rep, on=['Dataset', 'K'])

# 3. Calcul des écarts (Diff = Repro - Papier)
df_compare['Diff_CV'] = (df_compare['CV_rep'] - df_compare['CV_pap']).round(3)
df_compare['Diff_TD'] = (df_compare['TD_rep'] - df_compare['TD_pap']).round(3)
df_compare['Diff_Purity'] = (df_compare['Purity_rep'] - df_compare['Purity_pap']).round(3)
df_compare['Diff_NMI'] = (df_compare['NMI_rep'] - df_compare['NMI_pap']).round(3)

# Organisation des colonnes
final_cols = [
    'Dataset', 'K', 
    'CV_pap', 'CV_rep', 'Diff_CV',
    'TD_pap', 'TD_rep', 'Diff_TD',
    'Purity_pap', 'Purity_rep', 'Diff_Purity',
    'NMI_pap', 'NMI_rep', 'Diff_NMI'
]

print("="*125)
print(f"{'COMPARAISON FINALE HYPERMINER : PAPIER VS REPRODUCTION':^125}")
print("="*125)
print(df_compare[final_cols].to_string(index=False))

                                   COMPARAISON FINALE HYPERMINER : PAPIER VS REPRODUCTION                                    
    Dataset   K  CV_pap  CV_rep  Diff_CV  TD_pap  TD_rep  Diff_TD  Purity_pap  Purity_rep  Diff_Purity  NMI_pap  NMI_rep  Diff_NMI
       20NG  50   0.371   0.423    0.052   0.613   0.604   -0.009       0.433       0.544        0.111    0.405    0.546     0.141
       20NG 100   0.368   0.417    0.049   0.446   0.486    0.040       0.454       0.614        0.160    0.386    0.552     0.166
       IMDB  50   0.347   0.406    0.059   0.485   0.606    0.121       0.655       0.678        0.023    0.046    0.061     0.015
       IMDB 100   0.343   0.458    0.115   0.258   0.510    0.252       0.641       0.628       -0.013    0.032    0.038     0.006
YahooAnswer  50   0.344   0.383    0.039   0.507   0.624    0.117       0.456       0.570        0.114    0.237    0.334     0.097
YahooAnswer 100   0.346   0.326   -0.020   0.444   0.530    0.086       0.448       0.55